# MedFlow — 00 · Ingestão de dados (camada Bronze)

Este notebook **somente ingere e preserva** as fontes do DATASUS, Ministério
da Saúde e IBGE. Não aplica regra de negócio, de/para, filtro analítico,
imputação nem cálculo de indicador. Os únicos acréscimos são colunas técnicas
de linhagem e um manifesto com contagens, esquema e hashes.

**Entradas:** SIH/RD, CNES/LT, API de localidades do IBGE, API DEMAS de
regiões e estabelecimentos, CONCLA/IBGE e referência CID-10 do DATASUS.
**Saídas:** fontes imutáveis em `data/bronze/origem/`, cache DBF em
`data/bronze/intermediario/dbf/` e Parquets fiéis em `data/bronze/parquet/`.

Toda a lógica vive em `src/medflow/`, instalado por `make setup`. Este notebook
chama o pacote e mostra o resultado — ele é evidência, não é o motor. Rodar a
mesma coisa pela linha de comando:

```bash
medflow bronze
```

In [ ]:
from pathlib import Path

from medflow.bronze import executar

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SOBRESCREVER = False

In [ ]:
manifesto = executar(base=BASE, sobrescrever=SOBRESCREVER)

## Reconciliações do manifesto

As doze verificações abaixo são o que separa "baixou" de "baixou certo". Se
qualquer uma falhar, a execução acima já teria parado.

In [ ]:
import pandas as pd

pd.Series(manifesto["checks"]).to_frame("valor")

## Recorte efetivo e proveniência

O recorte é a interseção entre as competências publicadas no SIH/RD e no
CNES/LT — não o período solicitado. A diferença entre os dois é informação, e
fica registrada no manifesto.

In [ ]:
recorte = manifesto["recorte"]
print("solicitado:", " a ".join(recorte["periodo_solicitado"]))
print("efetivo   :", recorte["competencias"][0], "a", recorte["ultima_competencia_comum"])
print("competências:", len(recorte["competencias"]))
print()
for nome, item in manifesto["arquivos"].items():
    print(f'{nome:<32} {item["bytes"]:>13,} bytes  {item["sha256"][:16]}…')